# This notebook creates the $\varrho$-type matrices used in the $C_k$-equivariant models

# 90-degree rotations (for $C_4$ models)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import os
from collections import defaultdict
from PIL import Image
from scipy.sparse import lil_matrix

# Fixed rotation angle: π/2 (90° anticlockwise)
K = 4  # Four 90° rotations = 360°

def process_square_image_to_binary(filepath, size):
    """Convert square image to binary (0/1) and resize."""
    img = Image.open(filepath).convert('L')
    img = img.resize((size, size))
    arr = np.array(img)
    binary = (arr > 128).astype(np.float32)
    return binary

def get_square_concentric_rings(size):
    """
    Create concentric square rings where each ring's pixel count is a multiple of 4.
    For a square, we work from outer edge inward.
    """
    if size % 2 != 0:
        raise ValueError("Size must be even for symmetric square rings")
    
    perfect_rings = []
    
    # Process rings from outside to inside
    current_size = size
    offset = 0
    
    while current_size > 0:
        ring_pixels = []
        
        # Top edge (left to right)
        for j in range(offset, offset + current_size):
            ring_pixels.append((offset, j))
        
        # Right edge (top to bottom, excluding top-right corner)
        for i in range(offset + 1, offset + current_size):
            ring_pixels.append((i, offset + current_size - 1))
        
        # Bottom edge (right to left, excluding bottom-right corner)
        if current_size > 1:
            for j in range(offset + current_size - 2, offset - 1, -1):
                ring_pixels.append((offset + current_size - 1, j))
        
        # Left edge (bottom to top, excluding both corners)
        if current_size > 1:
            for i in range(offset + current_size - 2, offset, -1):
                ring_pixels.append((i, offset))
        
        # Only add ring if it has pixels and is a multiple of 4
        if len(ring_pixels) > 0:
            if len(ring_pixels) % 4 == 0:
                perfect_rings.append(ring_pixels)
                print(f"Ring at offset {offset}: size {len(ring_pixels)} = {len(ring_pixels)//4} × 4 ✓")
            else:
                # Trim to nearest multiple of 4
                target_size = (len(ring_pixels) // 4) * 4
                if target_size >= 4:
                    perfect_rings.append(ring_pixels[:target_size])
                    print(f"Ring at offset {offset}: trimmed to {target_size} = {target_size//4} × 4 ✓")
        
        # Move to next inner ring
        offset += 1
        current_size -= 2
    
    print(f"\nCreated {len(perfect_rings)} square concentric rings")
    return perfect_rings, size // 2

def create_square_90_degree_permutation(ring_pixels, center):
    """
    Create a 90° anticlockwise rotation permutation for square ring.
    For a square, 90° rotation maps each pixel to the next position in the ring.
    """
    m = len(ring_pixels)
    
    if m < 4 or m % 4 != 0:
        return {}
    
    # For 90° rotation, each pixel moves to position (i + m/4) mod m
    perm = {}
    shift = m // 4  # Quarter rotation
    
    for i, pixel in enumerate(ring_pixels):
        next_pixel = ring_pixels[(i + shift) % m]
        perm[pixel] = next_pixel
    
    return perm

def create_square_permutation_matrix_sparse(image, perfect_rings, center):
    """Create sparse permutation matrix G for 90° rotation where G^4 = I."""
    size = image.shape[0]
    n = size * size
    G = lil_matrix((n, n), dtype=np.int8)
    full_perm = {}
    
    for ring in perfect_rings:
        perm = create_square_90_degree_permutation(ring, center)
        if perm:
            full_perm.update(perm)
    
    # Build the permutation matrix
    fixed_count = 0
    moved_count = 0
    
    for i in range(size):
        for j in range(size):
            idx_from = i * size + j
            if (i, j) in full_perm:
                i_to, j_to = full_perm[(i, j)]
                idx_to = i_to * size + j_to
                G[idx_to, idx_from] = 1
                moved_count += 1
            else:
                # Fixed points
                G[idx_from, idx_from] = 1
                fixed_count += 1
    
    print(f"Permutation: {moved_count} pixels rotate, {fixed_count} pixels fixed")
    return G

def apply_square_rotation(image, perfect_rings, center, rotations):
    """Apply 90° rotation 'rotations' times."""
    rotated = np.copy(image)
    
    for ring in perfect_rings:
        perm = create_square_90_degree_permutation(ring, center)
        if not perm:
            continue
            
        # Apply permutation 'rotations' times
        for orig_pixel in perm:
            current_pixel = orig_pixel
            for _ in range(rotations):
                current_pixel = perm[current_pixel]
            rotated[current_pixel] = image[orig_pixel]
    
    return rotated

def plot_square_rotations(image, perfect_rings, center):
    """Plot original and all 4 rotations (0°, 90°, 180°, 270°, 360°)."""
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    rotations = [0, 1, 2, 3, 4]
    angles = [0, 90, 180, 270, 360]
    titles = ['Original\n(0°)', 'Rotation 1\n(90°)', 'Rotation 2\n(180°)', 
              'Rotation 3\n(270°)', 'Back to Start\n(360°)\nIDENTITY']
    colors = ['blue', 'black', 'black', 'black', 'red']
    
    for idx, (rot, angle, title, color) in enumerate(zip(rotations, angles, titles, colors)):
        rotated_image = apply_square_rotation(image, perfect_rings, center, rot)
        
        axes[idx].imshow(rotated_image, cmap='gray', vmin=0, vmax=1)
        axes[idx].set_title(title, fontsize=12, fontweight='bold', color=color)
        axes[idx].set_xticks([])
        axes[idx].set_yticks([])
        
        # Add border for original and identity
        if rot == 0 or rot == 4:
            for spine in axes[idx].spines.values():
                spine.set_color(color)
                spine.set_linewidth(3)
    
    plt.tight_layout()
    fig.suptitle('Square 90° Rotation Sequence: C_4 (G^4 = I)', 
                 fontsize=16, fontweight='bold', y=1.05)
    
    return fig

def plot_square_ring_visualization(perfect_rings, size):
    """Visualize the square concentric ring structure."""
    ring_image = np.zeros((size, size, 3))
    
    num_rings = len(perfect_rings)
    colors = plt.cm.tab20(np.linspace(0, 1, min(num_rings, 20)))
    
    if num_rings > 20:
        colors = plt.cm.viridis(np.linspace(0, 1, num_rings))
    
    for ring_idx, ring in enumerate(perfect_rings):
        color = colors[ring_idx % len(colors)]
        for i, j in ring:
            ring_image[i, j] = color[:3]
    
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(ring_image)
    ax.set_title(f'Square Concentric Ring Structure\n{len(perfect_rings)} Rings, Each with n × 4 Pixels', 
                 fontsize=16, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Add legend
    if num_rings <= 20:
        from matplotlib.patches import Patch
        legend_elements = []
        for i in range(min(10, num_rings)):
            ring_size = len(perfect_rings[i])
            legend_elements.append(
                Patch(color=colors[i], label=f'Ring {i}: {ring_size} px = {ring_size//4}×4')
            )
        if num_rings > 10:
            legend_elements.append(Patch(color='gray', label=f'... and {num_rings-10} more'))
        ax.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1.05, 0.5))
    
    plt.tight_layout()
    return fig

def verify_square_periodicity(perfect_rings, center):
    """Verify that G^4 = I for all square rings."""
    print("\n" + "="*60)
    print("SQUARE ROTATION PERIODICITY VERIFICATION")
    print("="*60)
    
    all_perfect = True
    total_pixels = 0
    perfect_pixels = 0
    
    for i, ring in enumerate(perfect_rings):
        m = len(ring)
        total_pixels += m
        
        if m % 4 != 0:
            print(f"Ring {i}: Size {m} not multiple of 4 ✗")
            all_perfect = False
            continue
            
        perm = create_square_90_degree_permutation(ring, center)
        if not perm:
            print(f"Ring {i}: Failed to create permutation ✗")
            all_perfect = False
            continue
            
        # Test: apply 4 times should return to original
        test_pixel = ring[0]
        current = test_pixel
        for step in range(4):
            current = perm[current]
        
        if test_pixel == current:
            multiple = m // 4
            print(f"Ring {i}: Size {m} = {multiple}×4 - G⁴ = I ✓")
            perfect_pixels += m
        else:
            print(f"Ring {i}: Periodicity FAILED ✗")
            all_perfect = False
    
    print("-" * 60)
    print(f"Perfect pixels: {perfect_pixels}/{total_pixels} ({perfect_pixels/total_pixels*100:.1f}%)")
    
    if all_perfect:
        print("SUCCESS: All rings have perfect 90° rotation periodicity!")
    else:
        print("WARNING: Some rings lack perfect periodicity")
    
    return all_perfect

# === Main ===
if __name__ == "__main__":
    N = 150  # Square image size (must be even)
    
    if N % 2 != 0:
        print("Warning: Image size should be even for symmetric rings")
        N = N + 1
    # Change to whatever image to test
    input_image = "../output_ellipses/SquareEllipse_Maj0.11Min0.01_SpacedOut0.5/p0.4_seed_330005_Q11_0.0499_Q12_0.2301.png"

    
    print("=" * 70)
    print("SQUARE IMAGE 90° ROTATION: C_4 (G^4 = I)")
    print("=" * 70)
    print(f"Fixed rotation angle: π/2 (90° anticlockwise)")
    
    # Process square image
    print(f"\n1. Loading {N}×{N} square image...")
    binary_image = process_square_image_to_binary(input_image, N)
    
    # Create square concentric rings
    print(f"\n2. Creating square concentric rings with n × 4 pixels...")
    perfect_rings, center = get_square_concentric_rings(N)
    
    # Create permutation matrix
    print(f"\n3. Building 90° rotation permutation matrix...")
    G_sparse = create_square_permutation_matrix_sparse(binary_image, perfect_rings, center)
    
    # Mathematical verification
    print(f"\n4. Verifying mathematical periodicity...")
    mathematical_correct = verify_square_periodicity(perfect_rings, center)
    
    # Visualize ring structure
    print(f"\n5. Visualizing square ring structure...")
    plot_square_ring_visualization(perfect_rings, N)
    
    # Plot rotations
    print(f"\n6. Generating rotation sequence...")
    fig_main = plot_square_rotations(binary_image, perfect_rings, center)
    
    # Empirical verification
    print(f"\n7. Empirical verification...")
    final_image = apply_square_rotation(binary_image, perfect_rings, center, 4)
    exact_match = np.allclose(final_image, binary_image)
    
    if exact_match:
        print("SUCCESS: 4 × 90° = 360° returns to original!")
    else:
        difference = np.sum(np.abs(final_image - binary_image))
        print(f"✗ Difference detected: total error = {difference}")
    
    # Final summary
    print("\n" + "=" * 70)
    if mathematical_correct and exact_match:
        print("SUCCESS: Perfect 90° square rotation!")
        print("   G^4 = I verified both mathematically and empirically")
    else:
        print("PARTIAL SUCCESS: Check ring construction")
    print("=" * 70)
    
    plt.show()

### Save as mtx:

In [ ]:
import os
from scipy.sparse import coo_matrix
from scipy.io import mmwrite

# Save the permutation matrix
try:
    # Ensure the ../Files directory exists
    os.makedirs("../Files", exist_ok=True)
    
    # Calculate angle in degrees
    angle =  360.0 / K
    
    # Convert sparse matrix to COO format
    if G_sparse is None:
        raise ValueError("G_sparse is None. Failed to create permutation matrix.")
    G_coo = G_sparse.tocoo()
    
    # Define output path
    output_path = f"../Files/BetterACpermutation_matrix_{N}_{angle:.3f}.mtx"
    
    # Save in Matrix Market format
    mmwrite(output_path, G_coo)
    absolute_path = os.path.abspath(output_path)
    print(f"Attempted to save permutation matrix to '{absolute_path}'")
    
    # Verify file exists
    if os.path.exists(absolute_path):
        print(f"Success: File confirmed to exist at '{absolute_path}'")
    else:
        print(f"Warning: File '{absolute_path}' was not found after saving.")
        
except Exception as e:
    print(f"Error saving permutation matrix: {str(e)}")

# $2\pi/ k$ rotations (for $C_k$ models)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import os
from collections import defaultdict
from PIL import Image
from scipy.sparse import lil_matrix

def process_image_to_binary(filepath, size):
    """Convert image to binary (0/1), resize, and apply circular mask."""
    img = Image.open(filepath).convert('L')
    img = img.resize((size, size))
    arr = np.array(img)
    binary = (arr > 128).astype(np.float32)
    mask = create_circular_mask(size)
    binary *= mask
    return binary

def create_circular_mask(size):
    """Create circular mask with center and radius."""
    Y, X = np.ogrid[:size, :size]
    center = size // 2
    dist_from_center = np.sqrt((X - center) ** 2 + (Y - center) ** 2)
    radius = center
    return (dist_from_center < radius).astype(np.float32)

def get_concentric_rings(size, k):
    """Create concentric rings where each ring's pixel count is a multiple of k, with pixels spread evenly."""
    center = size // 2
    max_radius = center
    
    # First, group pixels by their natural Euclidean ring and sort by angle
    natural_rings_with_angles = defaultdict(list)
    for i in range(size):
        for j in range(size):
            r = int(round(np.sqrt((i - center)**2 + (j - center)**2)))
            if r < max_radius and r > 0:  # Exclude center and outside
                # Calculate angle for sorting
                angle = math.atan2(-(i-center), j-center)  # -y for correct orientation
                natural_rings_with_angles[r].append((i, j, angle))
    
    # Sort each natural ring by angle
    for r in natural_rings_with_angles:
        natural_rings_with_angles[r].sort(key=lambda x: x[2])
        # Convert back to just coordinates but now sorted by angle
        natural_rings_with_angles[r] = [(x[0], x[1]) for x in natural_rings_with_angles[r]]
    
    # Now build concentric rings that are multiples of k
    perfect_rings = []
    current_ring_pixels = []
    used_pixels = set()
    
    # Process rings in order of increasing radius
    for r in sorted(natural_rings_with_angles.keys()):
        # Get pixels from this natural ring that haven't been used yet
        available_pixels = [p for p in natural_rings_with_angles[r] if p not in used_pixels]
        
        if not available_pixels:
            continue
            
        # Add these pixels to our current ring (they're already sorted by angle)
        current_ring_pixels.extend(available_pixels)
        
        # Check if current ring size is a multiple of k
        if len(current_ring_pixels) % k == 0 and len(current_ring_pixels) >= k:
            # We have a complete ring that's a multiple of k
            perfect_rings.append(current_ring_pixels.copy())
            used_pixels.update(current_ring_pixels)
            current_ring_pixels = []
        elif len(current_ring_pixels) > 2 * k:
            # We have more than 2k pixels but not a multiple
            # Find the largest multiple <= current size
            target_size = (len(current_ring_pixels) // k) * k
            if target_size >= k:
                # Take the target_size pixels that are most evenly distributed
                # by sampling at regular intervals from the sorted list
                sampled_pixels = []
                n_total = len(current_ring_pixels)
                for i in range(target_size):
                    # Sample at regular intervals to get even distribution
                    idx = int((i * n_total) / target_size) % n_total
                    sampled_pixels.append(current_ring_pixels[idx])
                
                # Remove duplicates and ensure we have exactly target_size unique pixels
                unique_sampled = []
                seen = set()
                for pixel in sampled_pixels:
                    if pixel not in seen and len(unique_sampled) < target_size:
                        unique_sampled.append(pixel)
                        seen.add(pixel)
                
                # If we don't have enough unique pixels, add the remaining ones in order
                if len(unique_sampled) < target_size:
                    for pixel in current_ring_pixels:
                        if pixel not in seen and len(unique_sampled) < target_size:
                            unique_sampled.append(pixel)
                            seen.add(pixel)
                
                perfect_rings.append(unique_sampled)
                used_pixels.update(unique_sampled)
                
                # Remove the used pixels from current_ring_pixels
                current_ring_pixels = [p for p in current_ring_pixels if p not in used_pixels]
    
    # Handle any remaining pixels in the final ring
    if len(current_ring_pixels) >= k:
        target_size = (len(current_ring_pixels) // k) * k
        if target_size >= k:
            # Evenly sample from the remaining pixels
            sampled_pixels = []
            n_total = len(current_ring_pixels)
            for i in range(target_size):
                idx = int((i * n_total) / target_size) % n_total
                sampled_pixels.append(current_ring_pixels[idx])
            
            # Remove duplicates
            unique_sampled = []
            seen = set()
            for pixel in sampled_pixels:
                if pixel not in seen and len(unique_sampled) < target_size:
                    unique_sampled.append(pixel)
                    seen.add(pixel)
            
            # Fill remaining slots if needed
            if len(unique_sampled) < target_size:
                for pixel in current_ring_pixels:
                    if pixel not in seen and len(unique_sampled) < target_size:
                        unique_sampled.append(pixel)
                        seen.add(pixel)
            
            perfect_rings.append(unique_sampled)
            used_pixels.update(unique_sampled)
    
    print(f"Created {len(perfect_rings)} concentric rings with sizes: {[len(ring) for ring in perfect_rings]}")
    
    # Verify all rings are multiples of k and report
    for i, ring in enumerate(perfect_rings):
        multiple = len(ring) // k
        if len(ring) % k == 0:
            print(f"Ring {i}: size {len(ring)} = {multiple} × {k} ✓")
        else:
            print(f"Ring {i}: size {len(ring)} NOT a multiple of {k} ✗")
    
    return perfect_rings, center

def create_perfect_cyclic_permutation(ring_pixels, center, k):
    """Create a perfect cyclic permutation where G^k = I exactly."""
    m = len(ring_pixels)
    
    if m < k:
        return {}
    
    # Verify we have a multiple of k
    if m % k != 0:
        return {}
    
    # Sort pixels by angle around center
    sorted_pixels = sorted(ring_pixels, key=lambda p: math.atan2(-(p[0]-center), p[1]-center))
    
    # Calculate the shift that gives us exact period k
    s = m // k
    
    # Create the cyclic permutation
    perm = {}
    for i, pixel in enumerate(sorted_pixels):
        next_pixel = sorted_pixels[(i + s) % m]
        perm[pixel] = next_pixel
    
    return perm

def create_perfect_permutation_matrix_sparse(image, perfect_rings, center, k):
    """Create sparse permutation matrix G where G^k = I exactly."""
    size = image.shape[0]
    n = size * size
    G = lil_matrix((n, n), dtype=np.int8)
    full_perm = {}
    
    for ring in perfect_rings:
        perm = create_perfect_cyclic_permutation(ring, center, k)
        if perm:
            full_perm.update(perm)
    
    # Build the permutation matrix
    fixed_count = 0
    moved_count = 0
    
    for i in range(size):
        for j in range(size):
            idx_from = i * size + j
            if (i, j) in full_perm:
                i_to, j_to = full_perm[(i, j)]
                idx_to = i_to * size + j_to
                G[idx_to, idx_from] = 1
                moved_count += 1
            else:
                # Fixed points: center pixel, outside mask, or pixels not in perfect rings
                G[idx_from, idx_from] = 1
                fixed_count += 1
    
    print(f"Permutation summary: {moved_count} pixels move, {fixed_count} pixels fixed")
    return G

def apply_perfect_rotation(image, perfect_rings, center, rotations, k):
    """Apply rotation using the perfect cyclic permutations."""
    rotated = np.copy(image)
    
    for ring in perfect_rings:
        perm = create_perfect_cyclic_permutation(ring, center, k)
        if not perm:
            continue
            
        # For each pixel in the ring, apply the permutation 'rotations' times
        for orig_pixel in perm:
            current_pixel = orig_pixel
            for _ in range(rotations):
                current_pixel = perm[current_pixel]
            rotated[current_pixel] = image[orig_pixel]
    
    return rotated

def plot_comprehensive_rotations(image, perfect_rings, center, k):
    """Create a comprehensive plot showing original and all rotations."""
    
    # Create all rotations from 0 to k
    rotations_to_show = list(range(k + 1))
    n_images = len(rotations_to_show)
    
    # Calculate grid dimensions
    n_cols = min(6, n_images)
    n_rows = (n_images + n_cols - 1) // n_cols
    
    # Create the figure
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    
    # Handle case when there's only one row or one column
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes
    elif n_cols == 1:
        axes = axes.reshape(-1)
    else:
        axes = axes.flatten()
    
    # Plot each rotation
    for idx, rotation_count in enumerate(rotations_to_show):
        if idx < len(axes):
            rotated_image = apply_perfect_rotation(image, perfect_rings, center, rotation_count, k)
            
            axes[idx].imshow(rotated_image, cmap='gray', vmin=0, vmax=1)
            
            # Calculate rotation angle in degrees
            rotation_deg = rotation_count * (360.0 / k)
            
            # Title formatting
            if rotation_count == 0:
                title = f"Original\n(0°)"
                color = 'blue'
            elif rotation_count == k:
                title = f"Rotation {k}\n({rotation_deg:.0f}°)\nIDENTITY ✓"
                color = 'red'
            else:
                title = f"Rotation {rotation_count}\n({rotation_deg:.0f}°)"
                color = 'black'
            
            axes[idx].set_title(title, fontsize=12, fontweight='bold', color=color)
            axes[idx].set_xticks([])
            axes[idx].set_yticks([])
            
            # Add a border to highlight important cases
            if rotation_count == 0:
                for spine in axes[idx].spines.values():
                    spine.set_color('blue')
                    spine.set_linewidth(3)
            elif rotation_count == k:
                for spine in axes[idx].spines.values():
                    spine.set_color('red')
                    spine.set_linewidth(3)
    
    # Hide any unused subplots
    for idx in range(len(rotations_to_show), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    
    # Add overall title
    fig.suptitle(f'Cyclic Group Action: C_{k} (G^{k} = I)\nConcentric Rings with n × {k} Pixels Each', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    return fig

def plot_ring_visualization(perfect_rings, center, size, k):
    """Visualize the concentric ring structure with unique colors for each ring."""
    ring_image = np.zeros((size, size, 3))
    
    num_rings = len(perfect_rings)
    
    # Generate unique colors for each ring
    if num_rings <= 20:
        # For small number of rings, use highly distinct colors
        colors = plt.cm.tab20(np.linspace(0, 1, num_rings))
    elif num_rings <= 50:
        # For medium number of rings, use tab20 extended with tab20b and tab20c
        colors_tab20 = plt.cm.tab20(np.linspace(0, 1, 20))
        colors_tab20b = plt.cm.tab20b(np.linspace(0, 1, min(20, num_rings - 20)))
        colors = np.vstack([colors_tab20, colors_tab20b])
        if num_rings > 40:
            colors_tab20c = plt.cm.tab20c(np.linspace(0, 1, min(20, num_rings - 40)))
            colors = np.vstack([colors, colors_tab20c])
        colors = colors[:num_rings]
    else:
        # For many rings, use Set3 which has good distinct colors
        # Repeat the colormap if needed to ensure uniqueness
        n_repeats = (num_rings + 11) // 12  # Set3 has 12 distinct colors
        colors = []
        for i in range(n_repeats):
            offset = i * 0.3 / n_repeats  # Slight offset for each repetition
            segment = plt.cm.Set3(np.linspace(offset, 1 - offset, 12))
            colors.extend(segment)
        colors = np.array(colors[:num_rings])
    
    # Ensure all colors are unique by checking for duplicates
    unique_colors = []
    for i, color in enumerate(colors):
        is_duplicate = False
        for j in range(i):
            if np.allclose(color, colors[j], atol=0.01):
                is_duplicate = True
                break
        if not is_duplicate:
            unique_colors.append(color)
        else:
            # Generate a new distinct color
            hue = (i * 0.618033988749895) % 1.0  # Golden ratio for good distribution
            new_color = plt.cm.hsv(hue)
            unique_colors.append(new_color)
    
    colors = np.array(unique_colors)[:num_rings]
    
    for ring_idx, ring in enumerate(perfect_rings):
        color = colors[ring_idx % len(colors)]
        for i, j in ring:
            ring_image[i, j] = color[:3]  # RGB only
    
    # Mark the center
    ring_image[center, center] = [1, 1, 1]  # White center
    
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(ring_image)
    ax.set_title(f'Concentric Ring Structure\n{len(perfect_rings)} Rings, Each with n × {k} Pixels', 
              fontsize=16, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Add colorbar to show the ring indices
    sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(0, num_rings-1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.8, aspect=20)
    cbar.set_label('Ring Index', rotation=270, labelpad=20)
    
    # Add a legend for the first 20 rings if we have a reasonable number
    if num_rings <= 30:
        from matplotlib.patches import Patch
        legend_elements = []
        for i in range(min(15, num_rings)):
            ring_size = len(perfect_rings[i])
            multiple = ring_size // k
            legend_elements.append(
                Patch(color=colors[i], 
                      label=f'Ring {i}: {ring_size} px = {multiple}×{k}')
            )
        
        if num_rings > 15:
            legend_elements.append(Patch(color='gray', label=f'... and {num_rings-15} more rings'))
        
        ax.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1.1, 0.5))
    
    plt.tight_layout()
    return fig

def verify_periodicity(perfect_rings, center, k):
    """Verify mathematically that G^k = I for all rings."""
    print("\n" + "="*60)
    print("MATHEMATICAL PERIODICITY VERIFICATION")
    print("="*60)
    
    all_perfect = True
    total_pixels = 0
    perfect_pixels = 0
    
    for i, ring in enumerate(perfect_rings):
        m = len(ring)
        total_pixels += m
        
        if m < k:
            print(f"Ring {i}: Size {m} < {k} - Too small ✗")
            all_perfect = False
            continue
            
        if m % k != 0:
            print(f"Ring {i}: Size {m} not multiple of {k} ✗")
            all_perfect = False
            continue
            
        perm = create_perfect_cyclic_permutation(ring, center, k)
        if not perm:
            print(f"Ring {i}: Failed to create permutation ✗")
            all_perfect = False
            continue
            
        # Test periodicity
        test_pixel = ring[0]
        current = test_pixel
        for step in range(k):
            current = perm[current]
        
        if test_pixel == current:
            multiple = m // k
            print(f"Ring {i}: Size {m} = {multiple}×{k} - G^{k} = I ✓")
            perfect_pixels += m
        else:
            print(f"Ring {i}: Size {m} - Periodicity FAILED ✗")
            all_perfect = False
    
    print("-" * 60)
    print(f"Perfect pixels: {perfect_pixels}/{total_pixels} ({perfect_pixels/total_pixels*100:.1f}%)")
    
    if all_perfect:
        print("ALL RINGS: Perfect periodicity achieved! G^k = I")
    else:
        print(" Some rings do not have perfect periodicity")
    
    return all_perfect

def plot_rotation_sequence(image, perfect_rings, center, k):
    """Plot a smooth sequence of rotations showing the cyclic nature."""
    # Create more intermediate steps for smooth animation preview
    n_steps = min(4 * k, 32)  # Show up to 32 steps max
    step_size = k / (n_steps // 4)
    
    rotations_to_show = [0]
    current_rotation = 0
    
    while current_rotation < k and len(rotations_to_show) < n_steps:
        current_rotation += step_size
        rotations_to_show.append(min(current_rotation, k))
    
    # Ensure we include the final identity rotation
    if rotations_to_show[-1] != k:
        rotations_to_show.append(k)
    
    n_images = len(rotations_to_show)
    n_cols = min(8, n_images)
    n_rows = (n_images + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes
    elif n_cols == 1:
        axes = axes.reshape(-1)
    else:
        axes = axes.flatten()
    
    for idx, rotation_count in enumerate(rotations_to_show):
        if idx < len(axes):
            rotated_image = apply_perfect_rotation(image, perfect_rings, center, int(rotation_count), k)
            
            axes[idx].imshow(rotated_image, cmap='gray', vmin=0, vmax=1)
            
            rotation_deg = rotation_count * (360.0 / k)
            
            if rotation_count == 0:
                title = f"Start\n0°"
            elif rotation_count == k:
                title = f"End\n{rotation_deg:.0f}°\nIdentity"
            else:
                title = f"Step {idx}\n{rotation_deg:.1f}°"
            
            axes[idx].set_title(title, fontsize=10)
            axes[idx].set_xticks([])
            axes[idx].set_yticks([])
    
    # Hide unused subplots
    for idx in range(len(rotations_to_show), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    fig.suptitle(f'Rotation Sequence: 0° to 360° in {k} Steps', fontsize=14, fontweight='bold', y=1.02)
    
    return fig

# === Main ===
if __name__ == "__main__":
    N = 150 # Image is N x N in size
    k = 16  # We want G^8 = I (rotation by 2π/8 = 45°)
    ## Note: for larger k, increase N (e.g., if k= 256 try N= 500+ for fewer aliasing effects)
    
    #input_image = "output_ellipses/Masked_Single_orientation_images/p1.0_seed_106_Q11_-0.4605_Q12_0.0971.png" # Test on some input image
    #input_image = "output_ellipses/EllipseWithDiffParametersMaj0.4Min0.04/p0.0_seed_1_Q11_-0.1034_Q12_-0.1301.png"
    input_image = "../../Images/p0.4_seed_1000428_Q11_0.0395_Q12_0.1949.png" # Test on some input image
    #input_image = "../Images/downsampled_49x49.png"  
    
    print("=" * 70)
    print(f"CONCENTRIC RING CYCLIC GROUP ACTION: C_{k} (G^{k} = I)")
    print("=" * 70)
    
    # Process image and create perfect concentric rings
    print(f"\n1. Creating concentric rings with n × {k} pixels...")
    binary_image = process_image_to_binary(input_image, N)
    perfect_rings, center = get_concentric_rings(N, k)
    
    # Create permutation matrix
    print(f"\n2. Building permutation matrix G for 2π/{k} rotation...")
    G_sparse = create_perfect_permutation_matrix_sparse(binary_image, perfect_rings, center, k)
    
    # Mathematical verification
    print(f"\n3. Verifying mathematical periodicity...")
    mathematical_correct = verify_periodicity(perfect_rings, center, k)
    
    # Visualize ring structure
    print(f"\n4. Visualizing concentric ring structure...")
    plot_ring_visualization(perfect_rings, center, N, k)
    
    # Plot comprehensive rotations
    print(f"\n5. Generating main rotation sequence...")
    fig_main = plot_comprehensive_rotations(binary_image, perfect_rings, center, k)
    
    # Plot smooth rotation sequence
    print(f"\n6. Generating smooth rotation sequence...")
    fig_sequence = plot_rotation_sequence(binary_image, perfect_rings, center, k)
    
    # Empirical verification
    print(f"\n7. Empirical verification...")
    final_image = apply_perfect_rotation(binary_image, perfect_rings, center, k, k)
    exact_match = np.allclose(final_image, binary_image)
    
    if exact_match:
        print(f" EMPIRICAL SUCCESS: {k} rotations return exactly to original!")
    else:
        difference = np.sum(np.abs(final_image - binary_image))
        print(f"  Small differences detected: total error = {difference}")
    
    # Final summary
    print("\n" + "=" * 70)
    if mathematical_correct and exact_match:
        print("COMPLETE SUCCESS: Perfect concentric ring construction!")
        print(f"   All rings have n × {k} pixels, G^{k} = I verified")
    else:
        print("PARTIAL SUCCESS: Some rings may not be perfect multiples")
    print("=" * 70)
    
    plt.show()

### Save as mtx:

In [ ]:
import os
from scipy.sparse import coo_matrix
from scipy.io import mmwrite

# Save the permutation matrix
try:
    # Ensure the ../Files directory exists
    os.makedirs("../Files", exist_ok=True)
    
    # Calculate angle in degrees
    angle = 360.0 / k
    
    # Convert sparse matrix to COO format
    if G_sparse is None:
        raise ValueError("G_sparse is None. Failed to create permutation matrix.")
    G_coo = G_sparse.tocoo()
    
    # Define output path
    output_path = f"../Files/BetterACpermutation_matrix_{N}_{angle:.3f}.mtx"
    
    # Save in Matrix Market format
    mmwrite(output_path, G_coo)
    absolute_path = os.path.abspath(output_path)
    print(f"Attempted to save permutation matrix to '{absolute_path}'")
    
    # Verify file exists
    if os.path.exists(absolute_path):
        print(f"Success: File confirmed to exist at '{absolute_path}'")
    else:
        print(f"Warning: File '{absolute_path}' was not found after saving.")
        
except Exception as e:
    print(f"Error saving permutation matrix: {str(e)}")